# Process PRISM secondary screen

In [ ]:
from pathlib import Path
import os

# set project root = parent of /notebooks
PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)

print("CWD set to:", Path.cwd())

imports + paths + file check

In [ ]:
from pathlib import Path
import pandas as pd

RAW = Path("data/raw")
OUT = Path("data/processed")
OUT.mkdir(parents=True, exist_ok=True)

prism_path = RAW / "secondary-screen-dose-response-curve-parameters.csv"
prism_path.exists(), prism_path

read + inspect columns

In [ ]:
pr = pd.read_csv(prism_path)
pr.shape, pr.columns[:25]

normalize + prefer PR500 if present

In [ ]:
if "name" in pr.columns and "compound_name" not in pr.columns:
    pr = pr.rename(columns={"name": "compound_name"})

if "row_name" in pr.columns:
    has_pr500 = pr["row_name"].astype(str).str.contains("PR500", na=False)
    if has_pr500.any():
        pr = pr.loc[has_pr500].copy()

keep = ["depmap_id", "broad_id", "compound_name", "auc", "ic50", "r2", "ec50", "slope"]
keep = [c for c in keep if c in pr.columns]
pr = pr[keep]

pr.head(), pr.shape

collapse duplicates (compound × cell line)

In [ ]:
num_cols = [c for c in pr.columns if c not in ["depmap_id", "broad_id", "compound_name"]]
agg = {c: "median" for c in num_cols}
agg["compound_name"] = "first"

pr2 = pr.groupby(["depmap_id", "broad_id"], as_index=False).agg(agg)

outpath = OUT / "prism_secondary_collapsed.parquet"
pr2.to_parquet(outpath, index=False)

print("Wrote:", outpath)
print("Rows:", f"{len(pr2):,}")
pr2.head()